# Phase 3 — Co-mutation lookup tool (Aim 1)

A clinician meets a patient with, say, lung cancer carrying *KRAS* G12C and asks four
questions. Today Jason answers them by searching GENIE by hand; this notebook answers them
directly.

| question | function | method (and why) |
|---|---|---|
| What else is usually altered with this? | `lookup()` | panel-aware counts in carriers vs non-carriers, by TMB group, with Phase 2's verdict |
| How common is this exact combination? | `combo_prevalence()` | exact counts among patients tested for every gene, with 95% intervals |
| Is this (rare) pair really over- or under-represented? | `test_pair()` | Phase 2's conditional test, run on demand for any pair -- including variants and pairs below Phase 2's filters |
| Given everything we know about this patient, what else is likely? | `predict_partners()` | one regularised logistic model per gene (chosen by testing, Section 5) |
| What does it mean for treatment? | `therapy_evidence()` | CIViC (open clinical-evidence knowledgebase), including evidence for combinations |

**Patients:** one sample per patient, with both mutation and copy-number data (so "not
altered" means both were checked). Statistical answers use non-hypermutated patients and
subtype strata, as Phase 2 does; counts are shown for every TMB group.

**Inputs:** `data/processed/` (Phase 1 and 2 outputs), `data/external/civic/` (CIViC nightly
evidence table), `scripts/validation/common.py` (loads the Phase 2 test so both phases run
the same code).

**Limits:** co-occurrence is not treatment prediction -- GENIE's main release has no
treatment or response data, so therapy information comes from CIViC, not from GENIE.

## 1. Load

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"
ULTRAMUTATED_TMB = 100        # same cut-offs as Phase 2
MIN_SUBTYPE_PATIENTS = 50

sys.path.insert(0, str(ROOT / "scripts" / "validation"))
from common import load_phase2_functions     # the exact Phase 2 test code
_p2 = load_phase2_functions()
ConditionalModel, pb_pair_test = _p2["ConditionalModel"], _p2["pair_test"]

clinical = pd.read_parquet(PROCESSED_DIR / "clinical_tidy.parquet")
clinical = clinical[clinical["IS_REPRESENTATIVE"] & clinical["CNA_TESTED"] & clinical["MUT_TESTED"]].set_index("SAMPLE_ID")
alterations = pd.read_parquet(
    PROCESSED_DIR / "alterations_long.parquet",
    columns=["Sample_ID", "Hugo_Symbol", "Alteration_Type", "HGVSp_Short", "CNA_Direction"])
alterations = alterations[alterations["Sample_ID"].isin(clinical.index)]

# A gene counts as tested only where the panel covers it for mutations AND copy number.
coverage = (pd.read_parquet(PROCESSED_DIR / "panel_gene_coverage.parquet")[["SEQ_ASSAY_ID", "Hugo_Symbol"]]
            .merge(pd.read_parquet(PROCESSED_DIR / "cna_gene_panel_coverage.parquet")))
phase2 = pd.read_parquet(PROCESSED_DIR / "comutation_pairs.parquet")

clinical["TMB_GROUP"] = np.select(
    [clinical["HYPERMUTATION_STATUS"] == "not_hypermutated",
     (clinical["HYPERMUTATION_STATUS"] == "hypermutated") & (clinical["TMB_MUT_PER_MB"] < ULTRAMUTATED_TMB),
     clinical["HYPERMUTATION_STATUS"] == "hypermutated"],
    ["non-hyper", "hyper", "ultra"], default="unknown")
# Subtype strata as in Phase 2: detailed type, or "other" if < 50 non-hypermutated patients.
_det = clinical["CANCER_TYPE_DETAILED"].fillna("NA").astype(str)
_n = _det[clinical["TMB_GROUP"] == "non-hyper"].groupby([clinical["CANCER_TYPE"], _det]).size()
clinical["SUBTYPE_STRATUM"] = [d if _n.get((c, d), 0) >= MIN_SUBTYPE_PATIENTS else "other"
                               for c, d in zip(clinical["CANCER_TYPE"], _det)]
print(f"{len(clinical):,} patients, {len(alterations):,} alteration rows")
print(clinical["TMB_GROUP"].value_counts().to_string())

138,206 patients, 1,055,981 alteration rows
TMB_GROUP
non-hyper    117562
hyper         15560
unknown        4605
ultra           479


## 2. Helpers

A *feature* is a gene (`"KRAS"`, any alteration) or a gene plus detail: a protein change
(`("KRAS", "p.G12C")`), `("MET", "AMP")` or `("CDKN2A", "DEL")`.

In [2]:
def as_feature(f) -> tuple[str, str | None]:
    return (f, None) if isinstance(f, str) else (f[0], f[1])


def label(f) -> str:
    g, d = as_feature(f)
    return g if d is None else f"{g} {d.replace('p.', '')}"


def carriers_of(sub_alt: pd.DataFrame, f) -> set:
    gene, detail = as_feature(f)
    hit = sub_alt["Hugo_Symbol"] == gene
    if detail in ("AMP", "DEL"):
        hit &= sub_alt["CNA_Direction"] == {"AMP": "Amplification", "DEL": "Deep_Deletion"}[detail]
    elif detail is not None:
        hit &= sub_alt["HGVSp_Short"] == detail
    return set(sub_alt.loc[hit, "Sample_ID"])


def tested_mask(patients: pd.DataFrame, gene: str) -> np.ndarray:
    return patients["SEQ_ASSAY_ID"].isin(coverage.loc[coverage["Hugo_Symbol"] == gene, "SEQ_ASSAY_ID"]).to_numpy()


def tested_counts(patients: pd.DataFrame) -> pd.Series:
    """Per gene: how many of these patients had that gene tested."""
    per_panel = patients["SEQ_ASSAY_ID"].value_counts()
    cov = coverage[coverage["SEQ_ASSAY_ID"].isin(per_panel.index)]
    return cov["SEQ_ASSAY_ID"].map(per_panel).groupby(cov["Hugo_Symbol"]).sum()


def altered_counts(sub_alt: pd.DataFrame, patients) -> pd.Series:
    return sub_alt[sub_alt["Sample_ID"].isin(patients)].groupby("Hugo_Symbol")["Sample_ID"].nunique()


def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.maximum(np.asarray(n, float), 1)
    p, denom = k / n, 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return np.clip(centre - half, 0, 1), np.clip(centre + half, 0, 1)


def patients_of(cancer_type: str, subtype: str | None = None) -> pd.DataFrame:
    p = clinical[clinical["CANCER_TYPE"] == cancer_type]
    return p if subtype is None else p[p["CANCER_TYPE_DETAILED"] == subtype]

## 3. `lookup()` — what else is altered with this alteration?

| column | meaning |
|---|---|
| `n_both / n_tested` | carriers who also have the partner / carriers tested for the partner |
| `pct_carriers` (95% CI) | how often the partner is altered **with** the query |
| `pct_non_carriers` | how often it is altered **without** it |
| `ratio` | carriers vs non-carriers |
| `pct_carriers_nonhyper` / `_hyper` | the same within TMB groups (hypermutated tumours inflate raw rates) |
| `phase2` | Phase 2's gene-level verdict: `candidate, replicated` / `candidate` / `significant` / `n.s.` / `not tested` |
| `check` | raw ratio points the other way from Phase 2: either the variant behaves unlike its gene (*MET* amplification vs *MET* mutations) or heavily altered patients inflate the raw rate -- trust Phase 2 for the gene, the non-hyper column for the size |

`subtype=` restricts to one detailed subtype (e.g. `"Lung Adenocarcinoma"`).

In [3]:
def lookup(cancer_type: str, query, subtype: str | None = None,
           min_both: int = 3, top: int | None = 15) -> pd.DataFrame:
    gene, _ = as_feature(query)
    ct = patients_of(cancer_type, subtype)
    sub_alt = alterations[alterations["Sample_ID"].isin(ct.index)]
    base = ct[tested_mask(ct, gene)]                 # only patients tested for the query gene
    carriers = carriers_of(sub_alt, query) & set(base.index)
    car, non = base.loc[sorted(carriers)], base.drop(index=list(carriers))
    print(f"{cancer_type}{' / ' + subtype if subtype else ''} | {label(query)}: "
          f"{len(car):,} of {len(base):,} tested patients ({len(car) / max(len(base), 1):.1%}); "
          "by TMB group: " + ", ".join(f"{k} {v:,}" for k, v in car["TMB_GROUP"].value_counts().items()))

    out = pd.DataFrame({"n_both": altered_counts(sub_alt, car.index), "n_tested": tested_counts(car)})
    out = out.drop(index=gene, errors="ignore").dropna()
    out = out[out["n_both"] >= min_both].astype(int)
    out["pct_carriers"] = out["n_both"] / out["n_tested"]
    out["ci_low"], out["ci_high"] = wilson(out["n_both"], out["n_tested"])
    out["pct_non_carriers"] = (altered_counts(sub_alt, non.index) / tested_counts(non)).reindex(out.index).fillna(0)
    out["ratio"] = out["pct_carriers"] / out["pct_non_carriers"].where(out["pct_non_carriers"] > 0)
    for grp, col in [("non-hyper", "pct_carriers_nonhyper"), ("hyper", "pct_carriers_hyper")]:
        g = car[car["TMB_GROUP"] == grp]
        out[col] = (altered_counts(sub_alt, g.index) / tested_counts(g)).reindex(out.index)

    p2 = phase2[(phase2["Cancer_Type"] == cancer_type) & ((phase2["Gene_A"] == gene) | (phase2["Gene_B"] == gene))]
    p2 = p2.assign(partner=p2["Gene_B"].where(p2["Gene_A"] == gene, p2["Gene_A"])).set_index("partner")
    verdict = np.select([p2["interaction_candidate"] & p2["replicated"], p2["interaction_candidate"], p2["q_value"] < 0.05],
                        ["candidate, replicated", "candidate", "significant"], "n.s.")
    out["phase2"] = pd.Series(verdict, index=p2.index).reindex(out.index).fillna("not tested")
    out["phase2_direction"] = p2["direction"].reindex(out.index).where(out["phase2"] != "not tested", "")
    out["phase2_q"] = p2["q_value"].reindex(out.index)
    this_dir = np.where(out["ratio"] > 1, "co-occurring", "exclusive")
    out["check"] = np.where(out["phase2"].isin(["candidate, replicated", "candidate", "significant"])
                            & (out["phase2_direction"] != this_dir), "differs from Phase 2", "")
    out = out.sort_values("n_both", ascending=False)
    return out if top is None else out.head(top)


def show(t: pd.DataFrame) -> pd.DataFrame:
    v = t.copy()
    for c in ["pct_carriers", "pct_non_carriers", "pct_carriers_nonhyper", "pct_carriers_hyper"]:
        v[c] = (100 * v[c]).round(1)
    v["95% CI"] = [f"{100 * lo:.0f}-{100 * hi:.0f}" for lo, hi in zip(v["ci_low"], v["ci_high"])]
    v["ratio"] = v["ratio"].round(2)
    v["phase2_q"] = v["phase2_q"].map(lambda q: "" if pd.isna(q) else f"{q:.1e}")
    return v[["n_both", "n_tested", "pct_carriers", "95% CI", "pct_non_carriers", "ratio",
              "pct_carriers_nonhyper", "pct_carriers_hyper", "phase2", "phase2_direction", "phase2_q", "check"]]

### Example — lung cancer, *KRAS* G12C

Literature check: STK11 in ~25% and KEAP1 in ~20% of *KRAS* G12C lung cancers (Skoulidis &
Heymach 2019); NTRK3 enriched with G12C (Scharpf et al. 2022, also GENIE).

In [4]:
show(lookup("Non-Small Cell Lung Cancer", ("KRAS", "p.G12C")))

Non-Small Cell Lung Cancer | KRAS G12C: 1,881 of 16,596 tested patients (11.3%); by TMB group: non-hyper 1,250, hyper 561, unknown 70


,n_both,n_tested,pct_carriers,95% CI,pct_non_carriers,ratio,pct_carriers_nonhyper,pct_carriers_hyper,phase2,phase2_direction,phase2_q,check
Hugo_Symbol,,,,,,,,,,,,
TP53,741,1860,39.8,38-42,55.0,0.72,31.3,56.9,"candidate, replicated",exclusive,1.5e-43,
STK11,486,1860,26.1,24-28,12.1,2.15,28.6,19.6,"candidate, replicated",co-occurring,8.2e-75,
KEAP1,353,1816,19.4,18-21,12.5,1.56,19.2,19.3,"candidate, replicated",co-occurring,6.1e-36,
ATM,269,1844,14.6,13-16,5.6,2.60,14.1,15.7,"candidate, replicated",co-occurring,2.7e-28,
RBM10,261,1667,15.7,14-17,8.9,1.75,13.8,21.6,"candidate, replicated",co-occurring,3.1e-05,
CDKN2A,258,1860,13.9,12-16,18.1,0.77,11.1,20.7,candidate,exclusive,6.1e-03,
SMARCA4,186,1860,10.0,9-11,7.6,1.31,8.6,13.0,"candidate, replicated",co-occurring,5.8e-10,
NKX2-1,175,1859,9.4,8-11,7.8,1.21,9.6,9.3,n.s.,co-occurring,7.2e-02,
CDKN2B,145,1859,7.8,7-9,10.4,0.75,7.0,10.2,candidate,exclusive,8.6e-06,


In [5]:
# Same query within adenocarcinoma only -- squamous lung cancer has different drivers.
show(lookup("Non-Small Cell Lung Cancer", ("KRAS", "p.G12C"), subtype="Lung Adenocarcinoma", top=8))

Non-Small Cell Lung Cancer / Lung Adenocarcinoma | KRAS G12C: 1,601 of 12,245 tested patients (13.1%); by TMB group: non-hyper 1,076, hyper 463, unknown 62


,n_both,n_tested,pct_carriers,95% CI,pct_non_carriers,ratio,pct_carriers_nonhyper,pct_carriers_hyper,phase2,phase2_direction,phase2_q,check
Hugo_Symbol,,,,,,,,,,,,
TP53,591,1583,37.3,35-40,49.2,0.76,28.7,55.5,"candidate, replicated",exclusive,1.5e-43,
STK11,409,1583,25.8,24-28,13.7,1.88,28.7,18.1,"candidate, replicated",co-occurring,8.2e-75,
KEAP1,288,1541,18.7,17-21,12.2,1.53,19.1,17.2,"candidate, replicated",co-occurring,6.1e-36,
ATM,234,1569,14.9,13-17,5.9,2.54,14.2,16.4,"candidate, replicated",co-occurring,2.7e-28,
RBM10,219,1403,15.6,14-18,10.2,1.52,14.2,20.7,"candidate, replicated",co-occurring,3.1e-05,
CDKN2A,197,1583,12.4,11-14,15.8,0.79,9.8,19.2,candidate,exclusive,6.1e-03,
NKX2-1,157,1582,9.9,9-11,9.6,1.04,9.9,10.4,n.s.,co-occurring,7.2e-02,
SMARCA4,148,1583,9.3,8-11,7.4,1.27,8.2,11.9,"candidate, replicated",co-occurring,5.8e-10,


## 4. `combo_prevalence()` — how common is this exact combination?

Patients carrying **all** listed alterations, among patients tested for **all** the genes,
overall and per TMB group.

In [6]:
def combo_prevalence(cancer_type: str, combo: list, subtype: str | None = None) -> pd.DataFrame:
    ct = patients_of(cancer_type, subtype)
    sub_alt = alterations[alterations["Sample_ID"].isin(ct.index)]
    base = ct[np.logical_and.reduce([tested_mask(ct, as_feature(f)[0]) for f in combo])]
    carriers = set(base.index)
    for f in combo:
        carriers &= carriers_of(sub_alt, f)
    rows = []
    for grp, pts in [("all", base)] + [(g, base[base["TMB_GROUP"] == g]) for g in ["non-hyper", "hyper", "ultra"]]:
        k = len(carriers & set(pts.index)); lo, hi = wilson(k, len(pts))
        rows.append({"group": grp, "n_with_combo": k, "n_tested": len(pts),
                     "pct": round(100 * k / max(len(pts), 1), 2), "95% CI": f"{100 * lo:.2f}-{100 * hi:.2f}"})
    print(f"{cancer_type}{' / ' + subtype if subtype else ''} | " + " + ".join(label(f) for f in combo))
    return pd.DataFrame(rows).set_index("group")

In [7]:
combo_prevalence("Non-Small Cell Lung Cancer", [("KRAS", "p.G12C"), "STK11", "KEAP1"])

Non-Small Cell Lung Cancer | KRAS G12C + STK11 + KEAP1


,n_with_combo,n_tested,pct,95% CI
group,,,,
all,236,15995,1.48,1.30-1.67
non-hyper,171,12008,1.42,1.23-1.65
hyper,57,3507,1.63,1.26-2.10
ultra,0,5,0.00,0.00-43.45


In [8]:
# A combination usually called mutually exclusive: rare, but it exists -- and is far more
# common in hypermutated tumours.
combo_prevalence("Colorectal Cancer", [("BRAF", "p.V600E"), "KRAS"])

Colorectal Cancer | BRAF V600E + KRAS


,n_with_combo,n_tested,pct,95% CI
group,,,,
all,10,12844,0.08,0.04-0.14
non-hyper,2,10298,0.02,0.01-0.07
hyper,6,2024,0.30,0.14-0.65
ultra,2,123,1.63,0.45-5.74


## 5. `test_pair()` — is a (rare) pair over- or under-represented?

Phase 2 only tests pairs passing its four filters; clinicians ask about rare pairs and about
specific variants. `test_pair()` runs **the same conditional test** for any two features in
one cancer type: non-hypermutated patients, gene rates fitted within subtypes, each patient
compared with their own alteration count (the genes of Phase 2's analysis for that cancer type
supply the counts). One question at a time needs no screen-wide correction, but a pair below
Phase 2's power gate (< 5 expected) is labelled *exploratory*.

**Check:** for pairs Phase 2 did test, `test_pair()` should give essentially the same answer.

In [9]:
def test_pair(cancer_type: str, a, b) -> dict:
    ct = patients_of(cancer_type)
    ct = ct[ct["TMB_GROUP"] == "non-hyper"]
    sub_alt = alterations[alterations["Sample_ID"].isin(ct.index)]
    p2 = phase2[phase2["Cancer_Type"] == cancer_type]
    queried = {as_feature(a)[0], as_feature(b)[0]}
    # Phase 2's genes for this cancer type (minus the queried genes, which enter as given).
    feats = [g for g in sorted(set(p2["Gene_A"]) | set(p2["Gene_B"])) if g not in queried] + [a, b]
    X = np.column_stack([ct.index.isin(carriers_of(sub_alt, f)) for f in feats])
    T = np.column_stack([tested_mask(ct, as_feature(f)[0]) for f in feats])
    model = ConditionalModel(X, T, ct["SUBTYPE_STRATUM"].to_numpy())
    i, j = len(feats) - 2, len(feats) - 1
    jt = T[:, i] & T[:, j]
    both = int((X[:, i] & X[:, j] & jt).sum())
    p, direction, expected = pb_pair_test(model.joint(i, j)[jt], both)
    return {"pair": f"{label(a)} + {label(b)}", "n_tested_both": int(jt.sum()), "n_both": both,
            "expected": round(expected, 1), "fold": round((both + 0.5) / (expected + 0.5), 2),
            "direction": direction, "p_value": p,
            "note": "exploratory: < 5 expected, low power" if expected < 5 else ""}


# Consistency with Phase 2 on pairs it tested
rows = []
for ct, a, b in [("Non-Small Cell Lung Cancer", "KRAS", "STK11"), ("Non-Small Cell Lung Cancer", "EGFR", "KRAS"),
                 ("Colorectal Cancer", "BRAF", "KRAS"), ("Breast Cancer", "PIK3CA", "PTEN")]:
    r = test_pair(ct, a, b)
    ref = phase2[(phase2["Cancer_Type"] == ct) & phase2["Gene_A"].isin([a, b]) & phase2["Gene_B"].isin([a, b])].iloc[0]
    rows.append({**r, "phase2_expected": round(ref["expected_both"], 1), "phase2_p": ref["p_value"]})
pd.DataFrame(rows)

,pair,n_tested_both,n_both,expected,fold,direction,p_value,note,phase2_expected,phase2_p
0,KRAS + STK11,12320,876,460.5,1.90,co-occurring,2.234960e-76,,460.4,2.013742e-76
1,EGFR + KRAS,12320,81,1085.5,0.08,exclusive,7.015732e-322,,1085.4,7.015732e-322
2,BRAF + KRAS,10298,59,367.6,0.16,exclusive,1.538334e-95,,374.4,8.513910e-99
3,PIK3CA + PTEN,12805,269,382.6,0.70,exclusive,1.854731e-10,,397.5,1.093063e-12


In [10]:
# Questions Phase 2 cannot answer: variant-level and rare pairs
pd.DataFrame([
    test_pair("Non-Small Cell Lung Cancer", ("KRAS", "p.G12C"), "STK11"),
    test_pair("Non-Small Cell Lung Cancer", ("KRAS", "p.G12D"), "STK11"),
    test_pair("Non-Small Cell Lung Cancer", ("EGFR", "p.L858R"), ("MET", "AMP")),
    test_pair("Non-Small Cell Lung Cancer", ("KRAS", "p.G12C"), ("EGFR", "p.L858R")),
    test_pair("Colorectal Cancer", ("BRAF", "p.V600E"), ("KRAS", "p.G12D")),
])

,pair,n_tested_both,n_both,expected,fold,direction,p_value,note
0,KRAS G12C + STK11,12320,358,183.2,1.95,co-occurring,3.034524e-32,
1,KRAS G12D + STK11,12320,106,77.5,1.37,co-occurring,2.086644e-03,
2,EGFR L858R + MET AMP,12320,43,29.5,1.45,co-occurring,2.220003e-02,
3,KRAS G12C + EGFR L858R,12320,2,149.9,0.02,exclusive,2.974215e-63,
4,BRAF V600E + KRAS G12D,10298,0,84.3,0.01,exclusive,5.531854e-38,


## 6. `predict_partners()` — given the whole profile, what else is likely?

The registered aim was to "match a new patient to genetically similar cases". Four ways to
use a patient's known alterations were compared on held-out patients (hide one real
alteration, rank all tested genes, AUROC of the hidden one;
`scripts/analyses/09_similarity_metrics.py`, `10_predict_partners.py`, 6 cancer types):

| method | AUROC |
|---|---|
| cancer-type frequency (no patient information) | 0.82-0.83 |
| nearest neighbours, Jaccard / rarity-weighted Jaccard / + same subtype | 0.81-0.82 |
| summed pairwise co-occurrence ratios | 0.77 |
| **one logistic model per gene** (other alterations + alteration load + subtype) | **0.855** |

Whole-profile similarity does not beat the base rate: panel profiles carry only 2-4
alterations, so "nearest neighbours" are noisy, and raw co-occurrence ratios are inflated by
heavily altered tumours (the same trap as Fisher's test). A model that sees all known
alterations jointly *and* the alteration load does beat it, in every cancer type. Exact
matches are still reported by `combo_prevalence()`.

In [11]:
from sklearn.linear_model import LogisticRegression

_models = {}


def _fit(cancer_type: str, min_altered: int = 20):
    """One L2-regularised logistic model per gene, on non-hypermutated patients of this type."""
    if cancer_type in _models:
        return _models[cancer_type]
    ct = patients_of(cancer_type); ct = ct[ct["TMB_GROUP"] == "non-hyper"]
    sub_alt = alterations[alterations["Sample_ID"].isin(ct.index)]
    freq = sub_alt.groupby("Hugo_Symbol")["Sample_ID"].nunique()
    genes = sorted(freq[freq >= min_altered].index)
    X = np.column_stack([ct.index.isin(carriers_of(sub_alt, g)) for g in genes]).astype(float)
    T = np.column_stack([tested_mask(ct, g) for g in genes])
    subtypes = sorted(ct["SUBTYPE_STRATUM"].unique())
    S = (ct["SUBTYPE_STRATUM"].to_numpy()[:, None] == np.array(subtypes)[None, :]).astype(float)
    load = np.log1p(X.sum(1, keepdims=True))
    models, rate = {}, X.sum(0) / np.maximum(T.sum(0), 1)
    for k, g in enumerate(genes):
        tr = T[:, k]
        if X[tr, k].sum() < 5:
            continue
        feats = np.hstack([np.delete(X, k, axis=1), np.log1p(np.delete(X, k, axis=1).sum(1, keepdims=True)), S])
        models[g] = LogisticRegression(C=0.5, max_iter=300).fit(feats[tr], X[tr, k])
    _models[cancer_type] = (genes, subtypes, models, dict(zip(genes, rate)))
    return _models[cancer_type]


def predict_partners(cancer_type: str, profile: list, subtype: str | None = None, top: int = 12) -> pd.DataFrame:
    """Probability of each other gene being altered, given the patient's known altered genes."""
    genes, subtypes, models, rate = _fit(cancer_type)
    known = {as_feature(f)[0] for f in profile}
    x = np.array([g in known for g in genes], float)
    s = np.array([subtype == st for st in subtypes], float)
    rows = []
    for k, g in enumerate(genes):
        if g in known or g not in models:
            continue
        xo = np.delete(x, k)
        p = models[g].predict_proba(np.hstack([xo, np.log1p(xo.sum()), s])[None, :])[0, 1]
        rows.append({"gene": g, "predicted": p, "base_rate": rate[g], "ratio": p / max(rate[g], 1e-6)})
    out = pd.DataFrame(rows).sort_values("predicted", ascending=False).head(top).set_index("gene")
    print(f"{cancer_type}{' / ' + subtype if subtype else ''} | known: {', '.join(sorted(known))}")
    return out.round(3)

In [12]:
predict_partners("Non-Small Cell Lung Cancer", ["KRAS", "STK11"], subtype="Lung Adenocarcinoma")

Non-Small Cell Lung Cancer / Lung Adenocarcinoma | known: KRAS, STK11


,predicted,base_rate,ratio
gene,,,
KEAP1,0.399,0.098,4.062
TP53,0.136,0.454,0.300
ATM,0.122,0.054,2.256
RBM10,0.103,0.090,1.142
SMARCA4,0.059,0.052,1.127
LRP1B,0.052,0.097,0.538
POLQ,0.034,0.044,0.767
NKX2-1,0.034,0.082,0.410
MYC,0.034,0.051,0.659


In [13]:
# Calibration on held-out patients: do predicted probabilities match observed rates?
def calibration(cancer_type: str, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    ct = patients_of(cancer_type); ct = ct[ct["TMB_GROUP"] == "non-hyper"]
    test = rng.random(len(ct)) < 0.2
    saved = _models.pop(cancer_type, None)
    global clinical
    full = clinical
    clinical = full.drop(index=ct.index[test])          # fit on the other 80% only
    try:
        genes, subtypes, models, rate = _fit(cancer_type)
    finally:
        clinical = full
        _models.pop(cancer_type, None)
        if saved: _models[cancer_type] = saved
    te = ct[test]; sub_alt = alterations[alterations["Sample_ID"].isin(te.index)]
    X = np.column_stack([te.index.isin(carriers_of(sub_alt, g)) for g in genes]).astype(float)
    T = np.column_stack([tested_mask(te, g) for g in genes])
    S = (te["SUBTYPE_STRATUM"].to_numpy()[:, None] == np.array(subtypes)[None, :]).astype(float)
    pred, obs = [], []
    for k, g in enumerate(genes):
        if g not in models: continue
        xo = np.delete(X, k, axis=1)
        p = models[g].predict_proba(np.hstack([xo, np.log1p(xo.sum(1, keepdims=True)), S]))[:, 1]
        pred.append(p[T[:, k]]); obs.append(X[T[:, k], k])
    pred, obs = np.concatenate(pred), np.concatenate(obs)
    bins = pd.cut(pred, [0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.4, 1.0])
    return (pd.DataFrame({"predicted": pred, "observed": obs}).groupby(bins, observed=True)
            .agg(n=("observed", "size"), mean_predicted=("predicted", "mean"), observed_rate=("observed", "mean")).round(3))


calibration("Non-Small Cell Lung Cancer")

,n,mean_predicted,observed_rate
"(0.0, 0.01]",644956,0.004,0.005
"(0.01, 0.02]",112498,0.014,0.014
"(0.02, 0.05]",68979,0.030,0.026
"(0.05, 0.1]",19627,0.068,0.057
"(0.1, 0.2]",8195,0.136,0.118
"(0.2, 0.4]",4353,0.283,0.263
"(0.4, 1.0]",5769,0.675,0.665


## 7. `therapy_evidence()` — what does it mean for treatment?

CIViC (civicdb.org) is an open, expert-curated knowledgebase of clinical evidence for
cancer variants (OncoKB needs a licence token; CIViC does not). Each evidence item links a
*molecular profile* -- a variant, or a **combination** such as "KRAS G12C AND STK11 Loss" --
to a disease, therapies, an evidence type (predictive / prognostic / diagnostic), a level
(A validated ... E inferential) and a citation. `therapy_evidence()` returns accepted
predictive and prognostic evidence for each queried alteration and for combinations of them,
evidence from the patient's cancer type first.

In [14]:
civic = pd.read_csv(ROOT / "data" / "external" / "civic" / "nightly-ClinicalEvidenceSummaries.tsv", sep="\t")
civic = civic[(civic["evidence_status"] == "accepted") & civic["evidence_type"].isin(["Predictive", "Prognostic"])]
print(f"CIViC: {len(civic):,} accepted predictive/prognostic evidence items, "
      f"{civic['molecular_profile'].nunique():,} molecular profiles")

DISEASE_PATTERN = {
    "Non-Small Cell Lung Cancer": "lung", "Colorectal Cancer": "colo|rect", "Breast Cancer": "breast",
    "Melanoma": "melanoma", "Pancreatic Cancer": "pancrea", "Glioma": "glio|astrocytoma|oligodendro",
    "Ovarian Cancer": "ovar", "Prostate Cancer": "prostat", "Bladder Cancer": "bladder|urothelial",
    "Endometrial Cancer": "endometri|uterine", "Thyroid Cancer": "thyroid",
    "Esophagogastric Cancer": "esophag|gastric|stomach", "Head and Neck Cancer": "head and neck",
    "Hepatobiliary Cancer": "cholangio|hepato|biliary", "Gastrointestinal Stromal Tumor": "gastrointestinal stromal"}


def civic_names(f) -> list[str]:
    gene, detail = as_feature(f)
    if detail == "AMP": return [f"{gene} Amplification"]
    if detail == "DEL": return [f"{gene} Deletion", f"{gene} Loss"]
    if detail is not None: return [f"{gene} {detail.replace('p.', '')}", f"{gene} Mutation"]
    return [f"{gene} Mutation", f"{gene} Amplification", f"{gene} Loss", f"{gene} Deletion"]


def therapy_evidence(cancer_type: str, profile: list, levels: str = "ABC") -> pd.DataFrame:
    names = {n for f in profile for n in civic_names(f)}
    genes = {as_feature(f)[0] for f in profile}
    single = civic["molecular_profile"].isin(names)
    # Combination profiles ("X AND Y") whose genes are all in the patient's profile
    combo = civic["molecular_profile"].str.contains(" AND ", na=False) & civic["molecular_profile"].map(
        lambda mp: all(part.strip().split(" ")[0] in genes for part in str(mp).split(" AND ")))
    ev = civic[(single | combo) & civic["evidence_level"].isin(list(levels))].copy()
    ev["kind"] = np.where(combo[ev.index], "combination", "single")
    pat = DISEASE_PATTERN.get(cancer_type)
    ev["this_cancer_type"] = ev["disease"].str.contains(pat, case=False, na=False) if pat else False
    ev = ev.sort_values(["this_cancer_type", "kind", "evidence_level"], ascending=[False, True, True])
    return ev[["molecular_profile", "kind", "disease", "this_cancer_type", "evidence_type", "evidence_level",
               "significance", "therapies", "citation", "evidence_civic_url"]].reset_index(drop=True)

CIViC: 3,433 accepted predictive/prognostic evidence items, 1,304 molecular profiles


In [15]:
ev = therapy_evidence("Non-Small Cell Lung Cancer", [("KRAS", "p.G12C"), "STK11", "KEAP1"])
print(f"{len(ev)} evidence items; {ev['this_cancer_type'].sum()} in lung cancer; "
      f"{(ev['kind'] == 'combination').sum()} for combinations")
ev[ev["this_cancer_type"]].head(15)

40 evidence items; 25 in lung cancer; 0 for combinations


,molecular_profile,kind,disease,this_cancer_type,evidence_type,evidence_level,significance,therapies,citation,evidence_civic_url
0,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,A,Resistance,"Selumetinib,Docetaxel","Jänne et al., 2017",https://civicdb.org/links/evidence_items/2998
1,KRAS G12C,single,Lung Non-small Cell Carcinoma,True,Predictive,A,Sensitivity/Response,Sotorasib,"Skoulidis et al., 2021",https://civicdb.org/links/evidence_items/9431
2,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Resistance,MEK Inhibitor RO4987655,"Zimmer et al., 2014",https://civicdb.org/links/evidence_items/997
3,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Sensitivity/Response,"Selumetinib,Docetaxel","Jänne et al., 2013",https://civicdb.org/links/evidence_items/999
4,KRAS G12C,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Sensitivity/Response,"Docetaxel,Selumetinib","Jänne et al., 2015",https://civicdb.org/links/evidence_items/1142
5,KRAS G12C,single,Lung Non-small Cell Carcinoma,True,Prognostic,B,Poor Outcome,NaN,"Ihle et al., 2012",https://civicdb.org/links/evidence_items/1217
6,KRAS Mutation,single,Lung Adenocarcinoma,True,Predictive,B,Sensitivity/Response,Salirasib,"Riely et al., 2011",https://civicdb.org/links/evidence_items/1218
7,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Sensitivity/Response,Trametinib,"Infante et al., 2012",https://civicdb.org/links/evidence_items/1220
8,KRAS Mutation,single,Lung Adenocarcinoma,True,Prognostic,B,Poor Outcome,NaN,"Johnson et al., 2013",https://civicdb.org/links/evidence_items/1700
9,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Resistance,Erlotinib,"Ramalingam et al., 2011",https://civicdb.org/links/evidence_items/2242


**A rare combination with combination-level evidence.** *BRAF* V600E and *NRAS* mutations
are strongly exclusive in melanoma, but the rare patients carrying both exist -- and CIViC
holds evidence for exactly that combination (resistance to BRAF/MEK inhibition).

In [16]:
mel = [("BRAF", "p.V600E"), "NRAS"]
display(combo_prevalence("Melanoma", mel))
display(pd.DataFrame([test_pair("Melanoma", *mel)]))
therapy_evidence("Melanoma", mel).query("kind == 'combination'")

Melanoma | BRAF V600E + NRAS


,n_with_combo,n_tested,pct,95% CI
group,,,,
all,7,4144,0.17,0.08-0.35
non-hyper,6,2220,0.27,0.12-0.59
hyper,1,1799,0.06,0.01-0.31
ultra,0,54,0.00,0.00-6.64


,pair,n_tested_both,n_both,expected,fold,direction,p_value,note
0,BRAF V600E + NRAS,2220,6,162.7,0.04,exclusive,1.658039e-71,


,molecular_profile,kind,disease,this_cancer_type,evidence_type,evidence_level,significance,therapies,citation,evidence_civic_url
0,BRAF V600E AND NRAS Mutation,combination,Melanoma,True,Predictive,B,Resistance,"Dabrafenib,Vemurafenib","Shi et al., 2014",https://civicdb.org/links/evidence_items/6263
1,BRAF V600E AND BRAF V600M,combination,Melanoma,True,Predictive,C,Sensitivity/Response,Dabrafenib,"Ponti et al., 2012",https://civicdb.org/links/evidence_items/73
2,BRAF V600 AND BRAF Amplification,combination,Melanoma,True,Predictive,C,Resistance,"Trametinib,Dabrafenib","Long et al., 2014",https://civicdb.org/links/evidence_items/7677


## 8. Do the tool's frequencies hold up in independent patients?

The tool's core numbers are conditional frequencies -- "of patients with A, x% also have B".
They should be the same in independent patients. Each frequency is computed separately in MSK
patients and in all other centres (different patients, panels and pipelines), for the 8 most
altered genes and their partners in 5 cancer types.

In [17]:
def centre_agreement(cancer_type: str, n_query: int = 8, min_both: int = 10) -> pd.DataFrame:
    ct = patients_of(cancer_type)
    sub_alt = alterations[alterations["Sample_ID"].isin(ct.index)]
    queries = sub_alt.groupby("Hugo_Symbol")["Sample_ID"].nunique().nlargest(n_query).index
    rows = []
    for q in queries:
        res = {}
        for side, pts in [("MSK", ct[ct["CENTER"] == "MSK"]), ("other", ct[ct["CENTER"] != "MSK"])]:
            base = pts[tested_mask(pts, q)]
            car = base.loc[sorted(carriers_of(sub_alt, q) & set(base.index))]
            res[side] = pd.DataFrame({"k": altered_counts(sub_alt, car.index), "n": tested_counts(car)}).dropna()
        m = res["MSK"].join(res["other"], lsuffix="_msk", rsuffix="_oth", how="inner").drop(index=q, errors="ignore")
        m = m[(m["k_msk"] >= min_both) & (m["k_oth"] >= min_both)]
        for partner, r in m.iterrows():
            lo1, hi1 = wilson(r["k_msk"], r["n_msk"]); lo2, hi2 = wilson(r["k_oth"], r["n_oth"])
            rows.append({"cancer_type": cancer_type, "query": q, "partner": partner,
                         "freq_msk": r["k_msk"] / r["n_msk"], "freq_other": r["k_oth"] / r["n_oth"],
                         "cis_overlap": bool(lo1 <= hi2 and lo2 <= hi1)})
    return pd.DataFrame(rows)


agree = pd.concat([centre_agreement(ct) for ct in
                   ["Non-Small Cell Lung Cancer", "Breast Cancer", "Colorectal Cancer", "Glioma", "Pancreatic Cancer"]])
summary = agree.groupby("cancer_type").apply(lambda d: pd.Series({
    "frequencies_compared": len(d), "correlation": np.corrcoef(d["freq_msk"], d["freq_other"])[0, 1],
    "median_abs_difference_pct": 100 * (d["freq_msk"] - d["freq_other"]).abs().median(),
    "95%_intervals_overlap": d["cis_overlap"].mean()}), include_groups=False).round(3)
summary

,frequencies_compared,correlation,median_abs_difference_pct,95%_intervals_overlap
cancer_type,,,,
Breast Cancer,1000.0,0.973,0.832,0.814
Colorectal Cancer,1846.0,0.974,0.578,0.895
Glioma,679.0,0.995,0.553,0.968
Non-Small Cell Lung Cancer,1460.0,0.893,0.798,0.799
Pancreatic Cancer,418.0,0.996,0.729,0.763


## 9. Worked example: one patient, all five questions

A lung adenocarcinoma with *KRAS* G12C and *STK11* loss-of-function -- a combination with
known clinical weight (poor immunotherapy response; Skoulidis & Heymach 2019).

In [18]:
patient = [("KRAS", "p.G12C"), "STK11"]
ct, sub = "Non-Small Cell Lung Cancer", "Lung Adenocarcinoma"
display(combo_prevalence(ct, patient, subtype=sub))
display(pd.DataFrame([test_pair(ct, *patient)]))
display(predict_partners(ct, ["KRAS", "STK11"], subtype=sub, top=8))
display(therapy_evidence(ct, patient).query("this_cancer_type").head(10))

Non-Small Cell Lung Cancer / Lung Adenocarcinoma | KRAS G12C + STK11


,n_with_combo,n_tested,pct,95% CI
group,,,,
all,403,12129,3.32,3.02-3.66
non-hyper,309,9423,3.28,2.94-3.66
hyper,84,2351,3.57,2.90-4.40
ultra,0,3,0.00,0.00-56.15


,pair,n_tested_both,n_both,expected,fold,direction,p_value,note
0,KRAS G12C + STK11,12320,358,183.2,1.95,co-occurring,3.034524e-32,


Non-Small Cell Lung Cancer / Lung Adenocarcinoma | known: KRAS, STK11


,predicted,base_rate,ratio
gene,,,
KEAP1,0.399,0.098,4.062
TP53,0.136,0.454,0.300
ATM,0.122,0.054,2.256
RBM10,0.103,0.090,1.142
SMARCA4,0.059,0.052,1.127
LRP1B,0.052,0.097,0.538
POLQ,0.034,0.044,0.767
NKX2-1,0.034,0.082,0.410


,molecular_profile,kind,disease,this_cancer_type,evidence_type,evidence_level,significance,therapies,citation,evidence_civic_url
0,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,A,Resistance,"Selumetinib,Docetaxel","Jänne et al., 2017",https://civicdb.org/links/evidence_items/2998
1,KRAS G12C,single,Lung Non-small Cell Carcinoma,True,Predictive,A,Sensitivity/Response,Sotorasib,"Skoulidis et al., 2021",https://civicdb.org/links/evidence_items/9431
2,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Resistance,MEK Inhibitor RO4987655,"Zimmer et al., 2014",https://civicdb.org/links/evidence_items/997
3,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Sensitivity/Response,"Selumetinib,Docetaxel","Jänne et al., 2013",https://civicdb.org/links/evidence_items/999
4,KRAS G12C,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Sensitivity/Response,"Docetaxel,Selumetinib","Jänne et al., 2015",https://civicdb.org/links/evidence_items/1142
5,KRAS G12C,single,Lung Non-small Cell Carcinoma,True,Prognostic,B,Poor Outcome,NaN,"Ihle et al., 2012",https://civicdb.org/links/evidence_items/1217
6,KRAS Mutation,single,Lung Adenocarcinoma,True,Predictive,B,Sensitivity/Response,Salirasib,"Riely et al., 2011",https://civicdb.org/links/evidence_items/1218
7,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Sensitivity/Response,Trametinib,"Infante et al., 2012",https://civicdb.org/links/evidence_items/1220
8,KRAS Mutation,single,Lung Adenocarcinoma,True,Prognostic,B,Poor Outcome,NaN,"Johnson et al., 2013",https://civicdb.org/links/evidence_items/1700
9,KRAS Mutation,single,Lung Non-small Cell Carcinoma,True,Predictive,B,Resistance,Erlotinib,"Ramalingam et al., 2011",https://civicdb.org/links/evidence_items/2242


## 10. Summary and limits

- **Lookup and combination counts** are panel-aware and split by TMB group; frequencies agree
  between MSK and the other centres (Section 8).
- **`test_pair()`** reproduces Phase 2 on pairs Phase 2 tested and extends the test to variants
  and rare pairs (labelled exploratory below 5 expected).
- **`predict_partners()`**: logistic models beat the base rate in every cancer type tested;
  similarity-based matching does not (Section 6).
- **`therapy_evidence()`**: CIViC, including combination evidence; disease matching is by
  keyword, so evidence from other cancer types is listed after the patient's own.
- **Limits:** GENIE has no treatment/response data, so nothing here predicts response;
  predictions are for non-hypermutated tumours; the logistic models are gene-level.